# Projeto: Consultor Imobiliario

Neste notebook vamos combinar todos os conceitos da aula 2 para construir um **consultor imobiliario inteligente**. O sistema usa:

- **Estado** para acumular as preferencias do usuario ao longo da conversa
- **Multi-agentes** especializados em busca de imoveis, analise de bairro e simulacao financeira
- **Tools** customizadas para calculo de financiamento e busca na web

A arquitetura segue o padrao coordenador + sub-agentes: o usuario conversa com o coordenador, que delega tarefas aos especialistas conforme necessario.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Ferramentas

Vamos definir duas ferramentas base: uma para busca na web (via Tavily) e outra para simulacao de financiamento imobiliario usando o sistema PRICE (parcelas fixas).

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def buscar_na_web(query: str) -> Dict[str, Any]:
    """Busca informacoes atualizadas na internet."""
    return tavily_client.search(query)

A tool de financiamento calcula parcelas fixas pelo sistema PRICE, o modelo mais comum no mercado brasileiro.

In [3]:
@tool
def simular_financiamento(valor_imovel: float, entrada_percentual: float, taxa_anual: float, prazo_meses: int) -> str:
    """Simula financiamento imobiliario pelo sistema PRICE (parcelas fixas)."""
    entrada = valor_imovel * (entrada_percentual / 100)
    financiado = valor_imovel - entrada
    taxa_mensal = taxa_anual / 100 / 12
    parcela = financiado * (taxa_mensal * (1 + taxa_mensal) ** prazo_meses) / ((1 + taxa_mensal) ** prazo_meses - 1)
    total_pago = entrada + (parcela * prazo_meses)
    return (
        f"Imovel: R$ {valor_imovel:,.2f} | Entrada ({entrada_percentual}%): R$ {entrada:,.2f}\n"
        f"Financiado: R$ {financiado:,.2f} | Taxa: {taxa_anual}% a.a. | Prazo: {prazo_meses} meses\n"
        f"Parcela: R$ {parcela:,.2f} | Total: R$ {total_pago:,.2f}"
    )

## Estado

O estado acumula as preferencias do usuario ao longo da conversa. Cada campo pode ser preenchido em momentos diferentes.

In [4]:
from langchain.agents import AgentState

class EstadoConsulta(AgentState):
    cidade: str
    bairro: str
    tipo_imovel: str
    orcamento: float
    numero_quartos: int

## Sub-agentes

Tres agentes especializados, cada um com um escopo bem definido. O **agente de imoveis** busca listagens disponiveis.

In [5]:
from langchain.agents import create_agent

agente_imoveis = create_agent(
    model="gpt-4.1-nano",
    tools=[buscar_na_web],
    system_prompt=(
        "Voce e um especialista em busca de imoveis. "
        "Use a ferramenta de busca para encontrar listagens de imoveis disponiveis "
        "com base nos criterios informados (cidade, bairro, tipo, orcamento, quartos). "
        "Apresente as opcoes de forma organizada."
    )
)

O **agente de bairro** pesquisa informacoes sobre qualidade de vida e infraestrutura.

In [6]:
agente_bairro = create_agent(
    model="gpt-4.1-nano",
    tools=[buscar_na_web],
    system_prompt=(
        "Voce e um especialista em analise de bairros. "
        "Use a ferramenta de busca para encontrar informacoes sobre qualidade de vida, "
        "seguranca, infraestrutura, transporte e lazer do bairro solicitado. "
        "Apresente um resumo objetivo."
    )
)

O **agente financeiro** simula financiamentos usando a ferramenta de calculo PRICE.

In [7]:
agente_financeiro = create_agent(
    model="gpt-4.1-nano",
    tools=[simular_financiamento],
    system_prompt=(
        "Voce e um especialista em financiamento imobiliario. "
        "Use a ferramenta de simulacao para calcular parcelas e custos. "
        "Se o usuario nao informar a taxa, use 10.5% ao ano como padrao. "
        "Se nao informar a entrada, use 20%. Se nao informar o prazo, use 360 meses."
    )
)

## Tools do coordenador

O coordenador tem quatro tools: uma para atualizar o estado com as preferencias do usuario, e tres para delegar aos sub-agentes.

In [8]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
def atualizar_consulta(
    cidade: str, bairro: str, tipo_imovel: str, orcamento: float, numero_quartos: int, runtime: ToolRuntime
) -> Command:
    """Atualiza as preferencias do cliente quando ele informar o que procura."""
    return Command(update={
        "cidade": cidade,
        "bairro": bairro,
        "tipo_imovel": tipo_imovel,
        "orcamento": orcamento,
        "numero_quartos": numero_quartos,
        "messages": [ToolMessage("Preferencias atualizadas.", tool_call_id=runtime.tool_call_id)]
    })

As tres tools de delegacao leem o estado e montam a pergunta adequada para cada sub-agente.

In [9]:
@tool
async def buscar_imoveis(runtime: ToolRuntime) -> str:
    """Busca imoveis disponiveis com base nas preferencias do cliente."""
    cidade = runtime.state.get("cidade", "")
    bairro = runtime.state.get("bairro", "")
    tipo = runtime.state.get("tipo_imovel", "")
    orcamento = runtime.state.get("orcamento", "")
    quartos = runtime.state.get("numero_quartos", "")
    pergunta = f"{tipo} de {quartos} quartos no {bairro}, {cidade}, ate R$ {orcamento}"
    resposta = await agente_imoveis.ainvoke({"messages": [HumanMessage(content=pergunta)]})
    return resposta["messages"][-1].content

As tools de pesquisa de bairro e simulacao financeira seguem o mesmo padrao.

In [10]:
@tool
async def pesquisar_bairro(runtime: ToolRuntime) -> str:
    """Pesquisa informacoes sobre o bairro de interesse do cliente."""
    bairro = runtime.state.get("bairro", "")
    cidade = runtime.state.get("cidade", "")
    pergunta = f"Qualidade de vida, seguranca e infraestrutura no {bairro}, {cidade}"
    resposta = await agente_bairro.ainvoke({"messages": [HumanMessage(content=pergunta)]})
    return resposta["messages"][-1].content

In [11]:
@tool
async def calcular_financiamento(runtime: ToolRuntime) -> str:
    """Simula o financiamento com base no orcamento do cliente."""
    orcamento = runtime.state.get("orcamento", 0)
    pergunta = f"Simule um financiamento para um imovel de R$ {orcamento}"
    resposta = await agente_financeiro.ainvoke({"messages": [HumanMessage(content=pergunta)]})
    return resposta["messages"][-1].content

## Coordenador

O agente coordenador recebe todas as tools e o `state_schema`. Ele decide autonomamente quando atualizar o estado e quando delegar aos especialistas.

In [12]:
from langgraph.checkpoint.memory import InMemorySaver

coordenador = create_agent(
    model="gpt-4.1-nano",
    tools=[atualizar_consulta, buscar_imoveis, pesquisar_bairro, calcular_financiamento],
    state_schema=EstadoConsulta,
    checkpointer=InMemorySaver(),
    system_prompt=(
        "Voce e um consultor imobiliario. Seu trabalho e ajudar o cliente a encontrar o imovel ideal.\n"
        "Quando o cliente informar suas preferencias, atualize o estado com a ferramenta atualizar_consulta.\n"
        "Use os especialistas para buscar imoveis, analisar bairros e simular financiamentos.\n"
        "Sempre consulte o estado antes de delegar, para passar os criterios corretos aos especialistas."
    )
)

config = {"configurable": {"thread_id": "consulta-1"}}

## Conversa multi-turno

Vamos simular uma consulta completa em quatro turnos. O estado acumula as preferencias e os sub-agentes sao acionados conforme necessario.

In [13]:
resposta = await coordenador.ainvoke(
    {"messages": [HumanMessage(content="Estou procurando um apartamento de 3 quartos no Leblon, Rio de Janeiro, orcamento de 1.5 milhao.")]},
    config
)

print(resposta["messages"][-1].content)

Gostaria de que eu verificasse a viabilidade de um financiamento para esse valor ou prefere alternativas dentro do seu orçamento exato?


In [14]:
resposta = await coordenador.ainvoke(
    {"messages": [HumanMessage(content="O que voce sabe sobre o Leblon? E um bom bairro para morar?")]},
    config
)

print(resposta["messages"][-1].content)

O Leblon, no Rio de Janeiro, é considerado um dos bairros mais valorizados e desejados da cidade. Ele oferece uma combinação única de tranquilidade, segurança, infraestrutura de alto padrão e opções de lazer. Além disso, possui praias famosas, como a Praia do Leblon, que atraem moradores e turistas. O bairro conta com uma excelente oferta de restaurantes, cafés, shoppings, escolas de qualidade e serviços de alto padrão, o que o torna uma excelente opção para quem busca qualidade de vida e conforto. 

A atmosfera do Leblon é mais tranquila e residencial, ideal para famílias, profissionais e pessoas que prezam por segurança e comodidade. No entanto, por ser uma região muito valorizada, o custo de vida e o valor dos imóveis também são mais elevados, o que deve ser considerado na decisão de morar lá.

Se desejar, posso ajudar a analisar se o bairro atende às suas necessidades e expectativas, ou fornecer mais detalhes específicos.


Agora vamos pedir uma simulacao financeira. O coordenador deve delegar ao agente financeiro, que vai usar o orcamento armazenado no estado.

In [15]:
resposta = await coordenador.ainvoke(
    {"messages": [HumanMessage(content="Simule um financiamento pra mim.")]},
    config
)

print(resposta["messages"][-1].content)

Com base nas condições atuais de financiamento, para um imóvel de R$ 1.500.000,00 com uma entrada de 20%, o valor financiado seria aproximadamente R$ 1.200.000,00. Com uma taxa de juros de 10,5% ao ano e um prazo de 30 anos, a parcela mensal ficaria em torno de R$ 10.977. Sem contar possíveis variações nas taxas ou condições específicas do banco, esse é um cenário estimado. Se desejar, posso ajudar a encontrar opções de bancos ou instituições financeiras para você obter uma cotação mais precisa.


Por fim, vamos confirmar que o estado persiste entre os turnos perguntando pelos criterios armazenados.

In [16]:
resposta = await coordenador.ainvoke(
    {"messages": [HumanMessage(content="Qual meu orcamento e quantos quartos eu quero?")]},
    config
)

print(resposta["messages"][-1].content)

Você mencionou que seu orçamento é de R$ 1.500.000 e que procura um apartamento com 3 quartos no bairro do Leblon, Rio de Janeiro. Se precisar de mais alguma informação ou desejar ajustar esses critérios, estou à disposição para ajudar!


Essa ultima pergunta confirma que o estado persiste entre os turnos. O coordenador lembra das preferencias definidas no primeiro turno sem precisar consultar o historico de mensagens.

Neste notebook combinamos **estado**, **multi-agentes** e **tools** em um sistema funcional. O coordenador gerencia a conversa, o estado acumula preferencias de forma estruturada, e os sub-agentes executam tarefas especializadas. Esse padrao arquitetural e a base para construir aplicacoes de IA mais complexas em producao.